In [1]:
import geopandas as gpd
import folium
from folium import plugins
import fiona
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

# Define paths
eez_folder = '/home/crimsondeepdarshak/Desktop/Deep_Darshak/References/Build_1_docs/Fishing_areas/World_EEZ_v12_20231025_gpkg'
eez_file = os.path.join(eez_folder, 'eez_v12.gpkg')
boundaries_file = os.path.join(eez_folder, 'eez_boundaries_v12.gpkg')

print("=" * 70)
print("WORLD EEZ DATA EXPLORATION")
print("=" * 70)

# Load EEZ polygons
print(f"\n📁 Loading EEZ Polygons from eez_v12.gpkg...")
gdf_eez = gpd.read_file(eez_file, layer='eez_v12')
print(f"✓ Loaded {len(gdf_eez)} EEZ zones")
print(f"  Columns: {gdf_eez.columns.tolist()}")
print(f"  CRS: {gdf_eez.crs}")
print(f"  Total area: {gdf_eez.geometry.area.sum():.2f} sq degrees")

print(f"\n📍 Sample EEZ data:")
print(gdf_eez[['GEONAME', 'TERRITORY1', 'SOVEREIGN1', 'AREA_KM2']].head(10))

# Load EEZ boundaries (line features)
print(f"\n📁 Loading EEZ Boundaries from eez_boundaries_v12.gpkg...")
gdf_boundaries = gpd.read_file(boundaries_file, layer='eez_boundaries_v12')
print(f"✓ Loaded {len(gdf_boundaries)} EEZ boundary lines")
print(f"  Columns: {gdf_boundaries.columns.tolist()}")
print(f"  CRS: {gdf_boundaries.crs}")

print(f"\n📍 Sample Boundary data:")
print(gdf_boundaries[['LINE_NAME', 'LINE_TYPE', 'TERRITORY1', 'TERRITORY2', 'LENGTH_KM']].head(10))

WORLD EEZ DATA EXPLORATION

📁 Loading EEZ Polygons from eez_v12.gpkg...
✓ Loaded 285 EEZ zones
  Columns: ['MRGID', 'GEONAME', 'MRGID_TER1', 'POL_TYPE', 'MRGID_SOV1', 'TERRITORY1', 'ISO_TER1', 'SOVEREIGN1', 'MRGID_TER2', 'MRGID_SOV2', 'TERRITORY2', 'ISO_TER2', 'SOVEREIGN2', 'MRGID_TER3', 'MRGID_SOV3', 'TERRITORY3', 'ISO_TER3', 'SOVEREIGN3', 'X_1', 'Y_1', 'MRGID_EEZ', 'AREA_KM2', 'ISO_SOV1', 'ISO_SOV2', 'ISO_SOV3', 'UN_SOV1', 'UN_SOV2', 'UN_SOV3', 'UN_TER1', 'UN_TER2', 'UN_TER3', 'geometry']
  CRS: EPSG:4326
  Total area: 16198.36 sq degrees

📍 Sample EEZ data:
                                             GEONAME  \
0  United States Exclusive Economic Zone (America...   
1        British Exclusive Economic Zone (Ascension)   
2  New Zealand Exclusive Economic Zone (Cook Isla...   
3  Overlapping claim Falkland / Malvinas Islands:...   
4  French Exclusive Economic Zone (French Polynesia)   
5         British Exclusive Economic Zone (Pitcairn)   
6     British Exclusive Economic Zone (Sa

/tmp/ipykernel_13891/1635291888.py:25: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  print(f"  Total area: {gdf_eez.geometry.area.sum():.2f} sq degrees")


In [ ]:
# Interactive Folium Map - EEZ Zones
print("\n" + "=" * 70)
print("CREATING INTERACTIVE FOLIUM MAPS")
print("=" * 70)

# Create base map for EEZ
m_eez = folium.Map(
    location=[20, 0],
    zoom_start=3,
    tiles='CartoDB positron'
)

print("\n🗺️  Creating EEZ interactive map...")

# Add EEZ zones with color gradient based on area
# Normalize area for color mapping
areas = gdf_eez['AREA_KM2'].fillna(0)
min_area = areas.min()
max_area = areas.max()

def get_color_for_area(area):
    """Return color based on EEZ area size"""
    if pd.isna(area) or area == 0:
        normalized = 0
    else:
        normalized = (area - min_area) / (max_area - min_area)
    
    # Color gradient: green (small) -> yellow -> red (large)
    if normalized < 0.33:
        return f'rgb(0, {int(255 * (normalized / 0.33))}, 0)'  # green
    elif normalized < 0.66:
        return f'rgb({int(255 * ((normalized - 0.33) / 0.33))}, 255, 0)'  # yellow
    else:
        return f'rgb(255, {int(255 * (1 - (normalized - 0.66) / 0.34))}, 0)'  # red

# Create feature group for EEZ zones
fg_eez = folium.FeatureGroup(name='EEZ Zones (Color by Area)', show=True)

for idx, row in gdf_eez.iterrows():
    color = get_color_for_area(row['AREA_KM2'])
    
    popup_html = f"""
    <div style="font-family: Arial; width: 280px;">
        <h4 style="margin-bottom: 5px; border-bottom: 2px solid #ccc; padding-bottom: 5px;">
            {row['GEONAME']}
        </h4>
        <table style="width: 100%; font-size: 12px;">
            <tr><td><b>Territory:</b></td><td>{row['TERRITORY1']}</td></tr>
            <tr><td><b>Sovereign:</b></td><td>{row['SOVEREIGN1']}</td></tr>
            <tr><td><b>Area (km²):</b></td><td>{row['AREA_KM2']:,.0f}</td></tr>
            <tr><td><b>ISO:</b></td><td>{row['ISO_TER1']}</td></tr>
        </table>
    </div>
    """
    
    folium.GeoJson(
        data=row.geometry.__geo_interface__,
        style_function=lambda x, color=color: {
            'fillColor': color,
            'color': '#333',
            'weight': 1,
            'opacity': 0.7,
            'fillOpacity': 0.5
        },
        highlight_function=lambda x: {
            'fillColor': '#ffff00',
            'color': '#000000',
            'weight': 2,
            'opacity': 1,
            'fillOpacity': 0.7
        },
        popup=folium.Popup(popup_html, max_width=350),
        tooltip=f"{row['GEONAME']} - {row['AREA_KM2']:,.0f} km²"
    ).add_to(fg_eez)

fg_eez.add_to(m_eez)

# Add EEZ boundaries
fg_boundaries = folium.FeatureGroup(name='EEZ Boundaries', show=False)

for idx, row in gdf_boundaries.iterrows():
    boundary_html = f"""
    <div style="font-family: Arial; width: 280px;">
        <h4 style="margin-bottom: 5px;">{row.get('LINE_NAME', 'Unknown')}</h4>
        <table style="width: 100%; font-size: 12px;">
            <tr><td><b>Type:</b></td><td>{row.get('LINE_TYPE', 'N/A')}</td></tr>
            <tr><td><b>Territory 1:</b></td><td>{row.get('TERRITORY1', 'N/A')}</td></tr>
            <tr><td><b>Territory 2:</b></td><td>{row.get('TERRITORY2', 'N/A')}</td></tr>
            <tr><td><b>Length (km):</b></td><td>{row.get('LENGTH_KM', 0):,.0f}</td></tr>
        </table>
    </div>
    """
    
    folium.GeoJson(
        data=row.geometry.__geo_interface__,
        style_function=lambda x: {
            'color': '#FF0000',
            'weight': 2,
            'opacity': 0.7
        },
        popup=folium.Popup(boundary_html, max_width=350),
        tooltip=f"{row.get('LINE_NAME', 'Boundary')} - {row.get('LENGTH_KM', 0):,.0f} km"
    ).add_to(fg_boundaries)

fg_boundaries.add_to(m_eez)

# Add layer control
folium.LayerControl(collapsed=False).add_to(m_eez)

# Add fullscreen button
plugins.Fullscreen(position='topright').add_to(m_eez)

# Save map
output_file = os.path.join(eez_folder, '../../FAO_EEZ_interactive.html')
m_eez.save(output_file)
print(f"✓ Interactive map saved to: FAO_EEZ_interactive.html")
m_eez


CREATING INTERACTIVE FOLIUM MAPS

🗺️  Creating EEZ interactive map...
✓ Interactive map saved to: FAO_EEZ_interactive.html


In [ ]:
# Statistical Analysis
print("\n" + "=" * 70)
print("EEZ STATISTICAL ANALYSIS")
print("=" * 70)

print(f"\n📊 EEZ ZONES OVERVIEW:")
print(f"  Total EEZ zones: {len(gdf_eez)}")
print(f"  Total area covered: {gdf_eez['AREA_KM2'].sum():,.0f} km²")
print(f"  Average EEZ area: {gdf_eez['AREA_KM2'].mean():,.0f} km²")
print(f"  Largest EEZ: {gdf_eez.loc[gdf_eez['AREA_KM2'].idxmax(), 'GEONAME']} ({gdf_eez['AREA_KM2'].max():,.0f} km²)")
print(f"  Smallest EEZ: {gdf_eez.loc[gdf_eez['AREA_KM2'].idxmin(), 'GEONAME']} ({gdf_eez['AREA_KM2'].min():,.0f} km²)")

print(f"\n📍 EEZ BOUNDARIES OVERVIEW:")
print(f"  Total boundary lines: {len(gdf_boundaries)}")
print(f"  Total boundary length: {gdf_boundaries['LENGTH_KM'].sum():,.0f} km")
print(f"  Average boundary length: {gdf_boundaries['LENGTH_KM'].mean():,.0f} km")

print(f"\n🏛️ TOP 15 LARGEST EEZ ZONES:")
top_eez = gdf_eez.nlargest(15, 'AREA_KM2')[['GEONAME', 'TERRITORY1', 'SOVEREIGN1', 'AREA_KM2']]
for idx, (i, row) in enumerate(top_eez.iterrows(), 1):
    print(f"  {idx:2d}. {row['GEONAME']:40s} | {row['AREA_KM2']:>12,.0f} km² | {row['SOVEREIGN1']}")

print(f"\n📊 Top 10 Countries/Territories by Total EEZ Area:")
territory_area = gdf_eez.groupby('SOVEREIGN1')['AREA_KM2'].sum().nlargest(10)
for idx, (territory, area) in enumerate(territory_area.items(), 1):
    print(f"  {idx:2d}. {territory:30s}: {area:>12,.0f} km²")

# Line type distribution
print(f"\n📏 EEZ BOUNDARY TYPES:")
line_types = gdf_boundaries['LINE_TYPE'].value_counts()
for line_type, count in line_types.items():
    print(f"  {line_type:25s}: {count:3d} lines")

In [ ]:
# Advanced Folium Map with Region Focus
print("\n" + "=" * 70)
print("REGIONAL ANALYSIS")
print("=" * 70)

# Focus on a specific region (e.g., Southeast Asia)
# Get bounds for this region
southeast_asia_bounds = {
    'minlon': 95,
    'maxlon': 145,
    'minlat': -15,
    'maxlat': 25
}

# Filter EEZ zones for the region
eez_region = gdf_eez.cx[southeast_asia_bounds['minlon']:southeast_asia_bounds['maxlon'], 
                         southeast_asia_bounds['minlat']:southeast_asia_bounds['maxlat']]

print(f"\n🗺️  Southeast Asia Region:")
print(f"  EEZ zones in region: {len(eez_region)}")
print(f"  Total area: {eez_region['AREA_KM2'].sum():,.0f} km²")
print(f"\n  EEZ in this region:")
for idx, row in eez_region.iterrows():
    print(f"    • {row['GEONAME']} ({row['AREA_KM2']:,.0f} km²)")

# Create regional map
m_region = folium.Map(
    location=[5, 120],
    zoom_start=5,
    tiles='CartoDB positron'
)

# Add regional EEZ zones
for idx, row in eez_region.iterrows():
    color = '#2E86AB'  # Blue
    
    popup_html = f"""
    <div style="font-family: Arial; width: 280px; background-color: #f9f9f9; padding: 10px; border-radius: 5px;">
        <h4 style="margin-bottom: 5px; color: #2E86AB;">{row['GEONAME']}</h4>
        <table style="width: 100%; font-size: 12px;">
            <tr><td><b>Territory:</b></td><td>{row['TERRITORY1']}</td></tr>
            <tr><td><b>Sovereign:</b></td><td>{row['SOVEREIGN1']}</td></tr>
            <tr><td><b>Area (km²):</b></td><td>{row['AREA_KM2']:,.0f}</td></tr>
        </table>
    </div>
    """
    
    folium.GeoJson(
        data=row.geometry.__geo_interface__,
        style_function=lambda x, color=color: {
            'fillColor': color,
            'color': '#1a1a1a',
            'weight': 2,
            'opacity': 0.8,
            'fillOpacity': 0.4
        },
        highlight_function=lambda x: {
            'fillColor': '#FFD700',
            'color': '#000000',
            'weight': 3,
            'opacity': 1,
            'fillOpacity': 0.6
        },
        popup=folium.Popup(popup_html, max_width=350),
        tooltip=f"{row['GEONAME']} - {row['AREA_KM2']:,.0f} km²"
    ).add_to(m_region)

folium.LayerControl().add_to(m_region)
plugins.Fullscreen(position='topright').add_to(m_region)

# Save regional map
regional_file = os.path.join(eez_folder, '../../EEZ_Southeast_Asia_regional.html')
m_region.save(regional_file)
print(f"\n✓ Regional map saved to: EEZ_Southeast_Asia_regional.html")
m_region